In [ ]:
from pathlib import Path
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "pinn_swe_reflections").is_dir():
            return candidate
    raise RuntimeError("Cannot find project root. Run this notebook from inside the repository.")


ROOT_DIR = find_project_root()

# Override without editing the notebook:
#   PINN_SWE_RUN_DIR=/path/to/training_results/<run> jupyter lab
RUN_DIR = Path(
    os.environ.get("PINN_SWE_RUN_DIR", ROOT_DIR / "training_results" / "baseline")
).expanduser().resolve()

BOUNDARY_KM = 800.0
TIME_WINDOWS_DAYS = [(0.0, 1.0), (1.0, 2.0), (2.0, None)]
SLICE_DAYS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

SAVE_OUTPUTS = True
FIG_DPI = 160
OUTPUT_DIR = ROOT_DIR / "analysis_outputs" / "single_run_diagnostics" / RUN_DIR.name
if SAVE_OUTPUTS:
    (OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Outputs: {OUTPUT_DIR if SAVE_OUTPUTS else 'disabled'}")

In [ ]:
def read_json(path, default=None):
    path = Path(path)
    if not path.is_file():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def load_npy(path, required=True):
    path = Path(path)
    if not path.is_file():
        if required:
            raise FileNotFoundError(path)
        return None
    return np.load(path, allow_pickle=True)


def as_float_array(values):
    return np.asarray(values, dtype=np.float64)


def save_figure(fig, stem):
    if SAVE_OUTPUTS:
        fig.savefig(OUTPUT_DIR / "figures" / f"{stem}.png", dpi=FIG_DPI, bbox_inches="tight")


def dimensionalize_prediction(raw, hp, scale_kind):
    raw = as_float_array(raw)
    if not hp.get("non_dimensionalization", False):
        return raw
    if scale_kind == "vertical":
        return raw * float(hp["vertical_length_scale"])
    if scale_kind == "velocity":
        return raw * float(hp["horizontal_length_scale"]) / float(hp["time_scale"])
    raise ValueError(f"Unknown scale kind: {scale_kind}")


def load_prediction(run_dir, hp, dimensional_name, raw_name, scale_kind):
    dimensional = load_npy(run_dir / dimensional_name, required=False)
    if dimensional is not None:
        return as_float_array(dimensional)
    raw = load_npy(run_dir / raw_name, required=True)
    return dimensionalize_prediction(raw, hp, scale_kind)


if not RUN_DIR.is_dir():
    raise FileNotFoundError(RUN_DIR)

hp = read_json(RUN_DIR / "Hyper_Parameter_Dictionary.json", default={})
display(pd.Series(hp, name="value").to_frame().head(60))

FIELD_SPECS = {
    "eta": {
        "label": "eta / zeta",
        "unit": "m",
        "reference": "exact_solution_h_values.npy",
        "prediction_dimensional": "dimensional_network_output_h_values.npy",
        "prediction_raw": "network_output_h_values.npy",
        "scale_kind": "vertical",
        "time_mesh": "dimensional_zeta_solution_time_mesh_grid.npy",
        "x_mesh": "dimensional_zeta_solution_x_mesh_grid.npy",
    },
    "u": {
        "label": "u",
        "unit": "m/s",
        "reference": "exact_solution_u_values.npy",
        "prediction_dimensional": "dimensional_network_output_u_values.npy",
        "prediction_raw": "network_output_u_values.npy",
        "scale_kind": "velocity",
        "time_mesh": "dimensional_u_solution_time_mesh_grid.npy",
        "x_mesh": "dimensional_u_solution_x_mesh_grid.npy",
    },
}

fields = {}
for variable, spec in FIELD_SPECS.items():
    reference = as_float_array(load_npy(RUN_DIR / spec["reference"]))
    prediction = load_prediction(
        RUN_DIR,
        hp,
        spec["prediction_dimensional"],
        spec["prediction_raw"],
        spec["scale_kind"],
    )
    if reference.shape != prediction.shape:
        raise ValueError(f"{variable}: reference shape {reference.shape} != prediction shape {prediction.shape}")
    time_mesh = as_float_array(load_npy(RUN_DIR / spec["time_mesh"]))
    x_mesh = as_float_array(load_npy(RUN_DIR / spec["x_mesh"]))
    fields[variable] = {
        "spec": spec,
        "reference": reference,
        "prediction": prediction,
        "diff": prediction - reference,
        "abs_error": np.abs(prediction - reference),
        "time_mesh": time_mesh,
        "x_mesh": x_mesh,
        "time_days": time_mesh / 86400.0,
        "x_km": x_mesh / 1000.0,
    }

shape_rows = []
for variable, field in fields.items():
    shape_rows.append({
        "variable": variable,
        "shape": field["reference"].shape,
        "time_days_min": float(np.nanmin(field["time_days"])),
        "time_days_max": float(np.nanmax(field["time_days"])),
        "x_km_min": float(np.nanmin(field["x_km"])),
        "x_km_max": float(np.nanmax(field["x_km"])),
    })
shape_df = pd.DataFrame(shape_rows)
display(shape_df)
if SAVE_OUTPUTS:
    shape_df.to_csv(OUTPUT_DIR / "tables" / "loaded_shapes.csv", index=False)

In [ ]:
def compute_metrics(reference, prediction, mask=None):
    reference = as_float_array(reference)
    prediction = as_float_array(prediction)
    if mask is not None:
        reference = reference[mask]
        prediction = prediction[mask]
    diff = prediction - reference
    l1_denom = np.sum(np.abs(reference))
    l2_denom = np.linalg.norm(reference.ravel())
    return {
        "relative_l1": np.nan if l1_denom == 0 else np.sum(np.abs(diff)) / l1_denom,
        "relative_l2": np.nan if l2_denom == 0 else np.linalg.norm(diff.ravel()) / l2_denom,
        "mse": np.mean(diff ** 2),
        "rmse": np.sqrt(np.mean(diff ** 2)),
        "mae": np.mean(np.abs(diff)),
        "max_abs": np.max(np.abs(diff)),
        "n_points": int(diff.size),
    }


def window_label(lo, hi):
    return f"{lo:g}-{hi:g}d" if hi is not None else f">={lo:g}d"


metric_rows = []
for variable, field in fields.items():
    ref = field["reference"]
    pred = field["prediction"]
    time_days = field["time_days"]
    x_km = field["x_km"]

    for region_name, region_mask in [
        ("full_domain", np.ones_like(ref, dtype=bool)),
        (f"near_boundary_abs_x_ge_{BOUNDARY_KM:g}km", np.abs(x_km) >= BOUNDARY_KM),
    ]:
        row = {
            "variable": variable,
            "region": region_name,
            "time_window": "all",
            "time_lo_days": np.nan,
            "time_hi_days": np.nan,
        }
        row.update(compute_metrics(ref, pred, region_mask))
        metric_rows.append(row)

        for lo, hi in TIME_WINDOWS_DAYS:
            time_mask = time_days >= lo
            if hi is not None:
                time_mask = time_mask & (time_days < hi)
            mask = region_mask & time_mask
            if not np.any(mask):
                continue
            row = {
                "variable": variable,
                "region": region_name,
                "time_window": window_label(lo, hi),
                "time_lo_days": lo,
                "time_hi_days": np.nan if hi is None else hi,
            }
            row.update(compute_metrics(ref, pred, mask))
            metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.sort_values(["variable", "region", "time_lo_days"], na_position="first"))
if SAVE_OUTPUTS:
    metrics_df.to_csv(OUTPUT_DIR / "tables" / "metrics_by_window_and_region.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
for ax, variable in zip(axes, ["eta", "u"]):
    subset = metrics_df[(metrics_df["variable"] == variable) & (metrics_df["region"] == "full_domain")]
    subset = subset[subset["time_window"] != "all"]
    ax.bar(subset["time_window"], subset["relative_l2"])
    ax.set_title(f"{FIELD_SPECS[variable]['label']} relative L2 by time window")
    ax.set_ylabel("relative L2")
    ax.grid(True, axis="y", alpha=0.25)
save_figure(fig, "relative_l2_by_time_window")
plt.show()

In [ ]:
def history_epochs(values, hp):
    values = np.asarray(values).reshape(-1)
    output_period = int(hp.get("console_output_period", hp.get("output_period", 1)) or 1)
    if len(values) == 0:
        return np.array([], dtype=int)
    epochs = np.arange(len(values), dtype=int) * output_period
    epochs[0] = 0
    max_epoch = hp.get("epochs")
    if max_epoch is not None and len(values) > 1:
        epochs[-1] = min(int(max_epoch), int(epochs[-1]))
    return epochs


LOSS_FILES = {
    "total": "total_MSE_over_training.npy",
    "pde": "MSE_symbolic_functions_over_training.npy",
    "initial": "MSE_initial_conditions_over_training.npy",
    "boundary": "MSE_boundary_conditions_over_training.npy",
    "numerical_mse_eta": "Relative_L2_Error_h_over_training.npy",
    "numerical_mse_u": "Relative_L2_Error_u_over_training.npy",
    "euler_raw": "MSE_euler_transition_over_training.npy",
    "euler_u_raw": "MSE_euler_transition_u_over_training.npy",
    "euler_eta_raw": "MSE_euler_transition_h_over_training.npy",
}

loss_rows = []
for loss_name, filename in LOSS_FILES.items():
    values = load_npy(RUN_DIR / filename, required=False)
    if values is None:
        continue
    values = as_float_array(values).reshape(-1)
    for epoch, value in zip(history_epochs(values, hp), values):
        loss_rows.append({"loss": loss_name, "epoch": int(epoch), "value": float(value)})

euler_weight = float(hp.get("model_kwargs", {}).get("euler_transition_weight", 0.0) or 0.0)
if euler_weight != 0.0:
    euler_values = load_npy(RUN_DIR / "MSE_euler_transition_over_training.npy", required=False)
    if euler_values is not None:
        euler_values = as_float_array(euler_values).reshape(-1)
        for epoch, value in zip(history_epochs(euler_values, hp), euler_values * euler_weight):
            loss_rows.append({"loss": "euler_weighted", "epoch": int(epoch), "value": float(value)})

loss_df = pd.DataFrame(loss_rows)
if loss_df.empty:
    warnings.warn("No saved loss histories found.")
else:
    last_loss_df = loss_df.sort_values("epoch").groupby("loss", as_index=False).tail(1).sort_values("loss")
    display(last_loss_df)
    if SAVE_OUTPUTS:
        loss_df.to_csv(OUTPUT_DIR / "tables" / "loss_histories.csv", index=False)
        last_loss_df.to_csv(OUTPUT_DIR / "tables" / "loss_last_values.csv", index=False)

    selected = [
        "total", "pde", "initial", "boundary",
        "euler_raw", "euler_weighted",
        "numerical_mse_eta", "numerical_mse_u",
    ]
    selected = [name for name in selected if name in set(loss_df["loss"])]
    ncols = 2
    nrows = int(np.ceil(len(selected) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.3 * nrows), squeeze=False, constrained_layout=True)
    for ax, loss_name in zip(axes.ravel(), selected):
        group = loss_df[loss_df["loss"] == loss_name].sort_values("epoch")
        ax.semilogy(group["epoch"], group["value"], marker="o", markersize=3, linewidth=1.4)
        ax.set_title(loss_name)
        ax.set_xlabel("epoch")
        ax.grid(True, which="both", alpha=0.25)
    for ax in axes.ravel()[len(selected):]:
        ax.axis("off")
    save_figure(fig, "loss_histories")
    plt.show()

print(f"Euler transition weight from hyperparameters: {euler_weight:g}")
print("Note: numerical_mse_* comes from files named Relative_L2_Error_*_over_training.npy, but those arrays store MSE values.")

In [ ]:
def plot_triptych(variable):
    field = fields[variable]
    spec = field["spec"]
    reference = field["reference"]
    prediction = field["prediction"]
    abs_error = field["abs_error"]
    x_axis = field["time_days"]
    y_axis = field["x_km"]
    vmax = max(np.nanmax(np.abs(reference)), np.nanmax(np.abs(prediction)))
    error_vmax = np.nanmax(abs_error)
    panels = [
        ("reference", reference, "RdBu_r", -vmax, vmax),
        ("prediction", prediction, "RdBu_r", -vmax, vmax),
        ("absolute error", abs_error, "magma", 0.0, error_vmax),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
    for ax, (title, values, cmap, vmin, vmax_panel) in zip(axes, panels):
        mesh = ax.pcolormesh(
            x_axis,
            y_axis,
            values,
            shading="auto",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax_panel,
            rasterized=True,
        )
        ax.set_title(title)
        ax.set_xlabel("time, days")
        ax.set_ylabel("x, km")
        fig.colorbar(mesh, ax=ax, label=spec["unit"])
    fig.suptitle(f"{RUN_DIR.name} - {spec['label']}", y=1.05)
    save_figure(fig, f"field_triptych_{variable}")
    return fig


for variable in fields:
    fig = plot_triptych(variable)
    plt.show()

In [ ]:
def physical_average_sea_level(hp):
    value = float(hp.get("average_sea_level", 100.0))
    if hp.get("non_dimensionalization", False):
        value *= float(hp.get("vertical_length_scale", 1.0))
    return value


def physical_gravity(hp):
    if hp.get("non_dimensionalization", False):
        L = float(hp.get("horizontal_length_scale", 1.0))
        T = float(hp.get("time_scale", 1.0))
        Z = float(hp.get("vertical_length_scale", 1.0))
        vertical_scaling_factor = float(hp.get("vertical_scaling_factor", 1.0))
        return vertical_scaling_factor * L ** 2 / (T ** 2 * Z)
    return float(hp.get("gravitational_acceleration", 9.81))


def integrate_x(values, x_m):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(values, x_m, axis=0)
    return np.trapz(values, x_m, axis=0)


eta = fields["eta"]
vel = fields["u"]
time_days = eta["time_days"][0, :]
x_eta_m = eta["x_mesh"][:, 0]
x_u_m = vel["x_mesh"][:, 0]
H = physical_average_sea_level(hp)
g = physical_gravity(hp)

energy_ref = 0.5 * H * integrate_x(vel["reference"] ** 2, x_u_m) + 0.5 * g * integrate_x(eta["reference"] ** 2, x_eta_m)
energy_pred = 0.5 * H * integrate_x(vel["prediction"] ** 2, x_u_m) + 0.5 * g * integrate_x(eta["prediction"] ** 2, x_eta_m)
mass_ref = integrate_x(eta["reference"], x_eta_m)
mass_pred = integrate_x(eta["prediction"], x_eta_m)
max_eta_ref = np.nanmax(np.abs(eta["reference"]), axis=0)
max_eta_pred = np.nanmax(np.abs(eta["prediction"]), axis=0)
max_u_ref = np.nanmax(np.abs(vel["reference"]), axis=0)
max_u_pred = np.nanmax(np.abs(vel["prediction"]), axis=0)

diagnostics_df = pd.DataFrame({
    "time_days": time_days,
    "energy_ref": energy_ref,
    "energy_pred": energy_pred,
    "energy_ratio_pred_to_ref": energy_pred / np.maximum(energy_ref, np.finfo(float).eps),
    "mass_ref": mass_ref,
    "mass_pred": mass_pred,
    "max_abs_eta_ref": max_eta_ref,
    "max_abs_eta_pred": max_eta_pred,
    "max_abs_u_ref": max_u_ref,
    "max_abs_u_pred": max_u_pred,
})
display(diagnostics_df.describe().T)
if SAVE_OUTPUTS:
    diagnostics_df.to_csv(OUTPUT_DIR / "tables" / "time_series_diagnostics.csv", index=False)

energy_norm = max(float(energy_ref[0]), np.finfo(float).eps)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
ax = axes[0, 0]
ax.plot(time_days, energy_ref / energy_norm, label="reference")
ax.plot(time_days, energy_pred / energy_norm, label="prediction")
ax.set_title("energy normalized by initial reference energy")
ax.set_xlabel("time, days")
ax.grid(True, alpha=0.25)
ax.legend()

ax = axes[0, 1]
ax.plot(time_days, diagnostics_df["energy_ratio_pred_to_ref"])
ax.set_title("energy prediction / reference")
ax.set_xlabel("time, days")
ax.grid(True, alpha=0.25)

ax = axes[1, 0]
ax.plot(time_days, mass_ref, label="reference")
ax.plot(time_days, mass_pred, label="prediction")
ax.set_title("mass proxy: integral eta dx")
ax.set_xlabel("time, days")
ax.grid(True, alpha=0.25)
ax.legend()

ax = axes[1, 1]
ax.plot(time_days, max_eta_ref, label="eta reference")
ax.plot(time_days, max_eta_pred, label="eta prediction")
ax.plot(time_days, max_u_ref, label="u reference")
ax.plot(time_days, max_u_pred, label="u prediction")
ax.set_title("max absolute amplitude")
ax.set_xlabel("time, days")
ax.grid(True, alpha=0.25)
ax.legend()

save_figure(fig, "energy_mass_amplitude_diagnostics")
plt.show()

print(f"Physical H used for energy: {H:g} m")
print(f"Physical g used for energy: {g:g} m/s^2")

In [ ]:
def rmse_over_time(field, mask=None):
    diff = field["diff"]
    if mask is None:
        return np.sqrt(np.mean(diff ** 2, axis=0))
    return np.sqrt(np.mean(diff[mask] ** 2, axis=0))


eta_boundary_mask = np.abs(fields["eta"]["x_km"][:, 0]) >= BOUNDARY_KM
u_boundary_mask = np.abs(fields["u"]["x_km"][:, 0]) >= BOUNDARY_KM

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
for ax, variable, boundary_mask in [
    (axes[0], "eta", eta_boundary_mask),
    (axes[1], "u", u_boundary_mask),
]:
    field = fields[variable]
    t = field["time_days"][0, :]
    ax.plot(t, rmse_over_time(field), label="full domain")
    ax.plot(t, rmse_over_time(field, boundary_mask), label=f"|x| >= {BOUNDARY_KM:g} km")
    ax.set_title(f"{field['spec']['label']} RMSE over time")
    ax.set_xlabel("time, days")
    ax.set_ylabel(field["spec"]["unit"])
    ax.grid(True, alpha=0.25)
    ax.legend()
save_figure(fig, "rmse_over_time_boundary_vs_full")
plt.show()

In [ ]:
def nearest_time_index(time_days_1d, day):
    return int(np.nanargmin(np.abs(time_days_1d - day)))


for variable, field in fields.items():
    t = field["time_days"][0, :]
    x = field["x_km"][:, 0]
    valid_days = [day for day in SLICE_DAYS if np.nanmin(t) <= day <= np.nanmax(t)]
    ncols = 2
    nrows = int(np.ceil(len(valid_days) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.2 * nrows), squeeze=False, constrained_layout=True)
    for ax, day in zip(axes.ravel(), valid_days):
        idx = nearest_time_index(t, day)
        ax.plot(x, field["reference"][:, idx], label="reference", linewidth=1.6)
        ax.plot(x, field["prediction"][:, idx], label="prediction", linewidth=1.3)
        ax.set_title(f"{field['spec']['label']} at t={t[idx]:.3f} d")
        ax.set_xlabel("x, km")
        ax.set_ylabel(field["spec"]["unit"])
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(valid_days):]:
        ax.axis("off")
    save_figure(fig, f"spatial_slices_{variable}")
    plt.show()